# 校园快递包裹分类识别 —— Kaggle 训练 Notebook

运行顺序：克隆项目 → 检查 GPU → 安装依赖 → 检查数据 → 整理公开数据 → 配置类别 → 合并数据 → 训练 → 评估 → 可选启动推理服务。

已完成的数据准备步骤会根据 `data/dataset/data.yaml` 自动跳过。

运行前确认 Kaggle 已启用 GPU 和 Internet。

## 步骤1：从 GitHub 克隆项目

In [2]:
import os
import subprocess
from pathlib import Path
import shutil
from tqdm import tqdm
import time
import threading

REPO_URL = "https://github.com/cui0122/campus-express-yolov8.git"
BRANCH = "master"
CODE_COMMIT = ""

CLONE_DIR = Path("/kaggle/working/campus-express-yolov8")

class CloneProgress:
    def __init__(self):
        self.progress = 0
        self.status = "准备中..."
        self.pbar = None
        
    def update_progress(self):
        """模拟进度更新"""
        stages = [
            (0, "初始化连接..."),
            (10, "连接仓库..."),
            (20, "下载对象..."),
            (40, "接收对象..."),
            (60, "解析对象..."),
            (80, "检出文件..."),
            (90, "完成配置..."),
            (95, "清理缓存..."),
            (100, "完成！")
        ]
        
        current_stage = 0
        for progress, status in stages:
            if self.progress < progress:
                # 模拟每个阶段的耗时
                time.sleep(0.3)
                self.progress = progress
                self.status = status
                if self.pbar:
                    self.pbar.update(progress - self.pbar.n)
                    self.pbar.set_postfix({"状态": status})
        
        # 最后完成
        if self.pbar:
            self.pbar.update(100 - self.pbar.n)
            self.pbar.set_postfix({"状态": "✅ 完成"})

# 确保工作目录存在
os.makedirs("/kaggle/working", exist_ok=True)
os.chdir("/kaggle/working")

# 清理旧目录
if CLONE_DIR.exists():
    print("🧹 清理旧目录...")
    with tqdm(total=100, desc="删除旧目录", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} {postfix}') as pbar:
        for i in range(10):
            time.sleep(0.05)
            pbar.update(10)
            pbar.set_postfix({"状态": f"删除中 {i+1}/10"})
    shutil.rmtree(CLONE_DIR)
    print("✅ 清理完成")

# 克隆进度管理
progress_manager = CloneProgress()

print("🚀 开始克隆仓库...")
progress_manager.pbar = tqdm(
    total=100, 
    desc="克隆进度", 
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}',
    postfix={"状态": "准备中..."}
)

# 在后台线程中更新进度
progress_thread = threading.Thread(target=progress_manager.update_progress)
progress_thread.start()

# 执行克隆命令
clone_cmd = ["git", "clone", "-b", BRANCH, "--single-branch", "--progress", REPO_URL, str(CLONE_DIR)]
try:
    result = subprocess.run(clone_cmd, check=True, capture_output=True, text=True)
    print("\n✅ 克隆成功！")
except subprocess.CalledProcessError as e:
    print(f"\n❌ 克隆失败: {e.stderr}")
    raise

# 等待进度线程完成
progress_manager.pbar.set_postfix({"状态": "完成"})
progress_manager.pbar.close()
progress_thread.join()

# 切换到克隆的目录
os.chdir(CLONE_DIR)

if CODE_COMMIT:
    print(f"🔀 切换到commit: {CODE_COMMIT}")
    subprocess.run(["git", "checkout", CODE_COMMIT], check=True)

current_commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print(f"\n📁 项目目录: {CLONE_DIR}")
print(f"📌 当前代码版本: {current_commit}")

# 显示目录内容（带进度）
print("\n📂 目录内容:")
files = list(CLONE_DIR.iterdir())
with tqdm(total=len(files), desc="文件列表", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}') as pbar:
    for item in files:
        pbar.update(1)
        print(f"  📄 {item.name}")

🧹 清理旧目录...


删除旧目录: 100%|██████████| 100/100 , 状态=删除中 10/10


✅ 清理完成
🚀 开始克隆仓库...


克隆进度: 100%|██████████| 100/100 [00:24<00:00] , 状态=完成      



✅ 克隆成功！

📁 项目目录: /kaggle/working/campus-express-yolov8
📌 当前代码版本: d946daf

📂 目录内容:


文件列表: 100%|██████████| 11/11

  📄 README.md
  📄 .gitattributes
  📄 data
  📄 training
  📄 main.ipynb
  📄 system
  📄 requirements.txt
  📄 docs
  📄 .git
  📄 .vscode
  📄 .gitignore


## 训练前置状态总览

检查 `data/dataset/data.yaml` 是否已经存在（意味着合并/清洗/划分/增强全部跑完了）。
存在的话，下面第4~7步会自动全部跳过，直接从「阶段1训练」开始运行即可。


In [3]:
from pathlib import Path

DATASET_YAML = Path("data/dataset/data.yaml")
READY_FOR_TRAINING = DATASET_YAML.exists()

if READY_FOR_TRAINING:
    print(f"检测到已完成的数据集: {DATASET_YAML}")
    print(DATASET_YAML.read_text(encoding="utf-8"))
else:
    print("未检测到已完成的数据集，将执行数据准备流程。")

未检测到已完成的数据集，将执行数据准备流程。


## 步骤2：检查 GPU

In [4]:
!nvidia-smi

Wed Aug 19 01:49:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 步骤3：安装依赖

In [5]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.5 MB/s eta 0:00:00


## 步骤4：检查数据是否已就位

公开数据放在 `data/public_raw`，已整理的数据放在 `data/public_yolo`，校园数据放在 `data/campus_raw`。

In [8]:
from pathlib import Path

def exists(path, desc):
    path = Path(path)
    status = path.exists()
    print(f"{'✓' if status else '✗'} {desc}: {path}")
    return status

has_public_yolo = exists("data/public_yolo/images", "公开 YOLO 数据集")
has_public_raw = exists("data/public_raw", "公开原始数据集")

if has_public_yolo:
    SKIP_PREPARE = True
elif has_public_raw:
    SKIP_PREPARE = False
else:
    SKIP_PREPARE = None

✗ 公开 YOLO 数据集: data/public_yolo/images
✓ 公开原始数据集: data/public_raw


## 步骤5：整理公开数据集为统一 YOLO 格式（自动判断是否需要跑）

In [9]:
if READY_FOR_TRAINING:
    print("数据集已就绪，跳过公开数据整理。")
elif SKIP_PREPARE is None:
    raise FileNotFoundError("未检测到 data/public_yolo 或 data/public_raw。")
elif SKIP_PREPARE:
    print("公开数据集已整理，跳过此步骤。")
else:
    !python data/scripts/prepare_public_dataset.py --src data/public_raw --out data/public_yolo

检测到原始类别: ['Boxes', 'Parcel', 'good-parcel', 'label', 'package', 'parcel']
类别映射结果: {'Boxes': '纸箱', 'Parcel': '纸箱', 'good-parcel': '纸箱', 'package': '纸箱', 'parcel': '纸箱'}
⚠️ 以下原始类别未在 CLASS_MAP 中映射到任何最终类别，对应的框会被丢弃: ['label']
整理 test 子集: 100%|███████████████████████| 183/183 [00:00<00:00, 3135.27it/s]

[清理过滤统计]
🗑️ 自动彻底删除/剔除混入的前缀图片及标注: 0 张

[各最终类别实际保留框数统计]
  纸箱: 3679  ✅
  塑料袋: 0  ⚠️ 0个
  泡沫箱: 0  ⚠️ 0个

[OK] 共整理有效图片 2809 张 -> data/public_yolo


## 步骤6：配置类别映射

最终类别：纸箱、塑料袋、泡沫箱。

In [10]:
if READY_FOR_TRAINING:
    print("数据集已就绪，跳过类别映射配置。")
else:
    FINAL_CLASSES = ["纸箱", "塑料袋", "泡沫箱"]
    PUBLIC_CLASS_MAP = {
        "box": "纸箱",
        "boxes": "纸箱",
        "cardboard": "纸箱",
        "cardboard box": "纸箱",
        "corrugated carton": "纸箱",
        "package": "纸箱",
        "parcel": "纸箱",
        "plastic bag": "塑料袋",
        "single-use carrier bag": "塑料袋",
        "polypropylene bag": "塑料袋",
        "courier bag": "塑料袋",
        "poly mailer": "塑料袋",
        "foam food container": "泡沫箱",
        "styrofoam": "泡沫箱",
        "styrofam piece": "泡沫箱",
        "foam box": "泡沫箱",
        "label": None,
        "person": None,
    }

    script = Path("data/scripts/merge_datasets.py")
    content = script.read_text(encoding="utf-8")

    start = content.index("FINAL_CLASSES =")
    end = content.index("\n\n", start)
    config = (
        f"FINAL_CLASSES = {FINAL_CLASSES!r}\n\n"
        f"PUBLIC_CLASS_MAP = {PUBLIC_CLASS_MAP!r}"
    )
    script.write_text(content[:start] + config + content[end:], encoding="utf-8")
    print(f"类别映射已写入: {script}")

类别映射已写入: data/scripts/merge_datasets.py


## 步骤7：合并 + 清洗 + 划分 + 增强（数据集已就绪时自动跳过）

In [11]:
if READY_FOR_TRAINING:
    print("数据集已就绪，跳过合并、清洗、划分和增强。")
else:
    for script in [
        "merge_datasets.py",
        "clean_dataset.py",
        "split_dataset.py",
        "augment.py",
    ]:
        subprocess.run(["python", f"data/scripts/{script}"], check=True)

[campus] 合并数据集:  34%|███▍      | 360/1050 [00:00<00:00, 3598.25it/s]

[pub] 保留 2225 张，跳过 584 张


[campus] 合并数据集: 100%|██████████| 1050/1050 [00:00<00:00, 3442.46it/s]


[campus] 保留 1050 张，跳过 0 张
[OK] 合并完成 -> data/merged（下一步跑 clean_dataset.py）


1/4 计算感知哈希:   0%|          | 0/3275 [00:00<?, ?it/s]

...

train: 2062 张（占比 70.0%，目标 70%）
val: 589 张（占比 20.0%，目标 20%）


写入 test 集: 100%|██████████| 295/295 [00:00<00:00, 3957.38it/s]


test: 295 张（占比 10.0%，目标 10%）
[OK] 划分完成，data.yaml 已生成: data/dataset/data.yaml
待增强图像数: 2062


执行 train 集数据增强: 100%|██████████| 2062/2062 [04:01<00:00,  8.54it/s]


[OK] 增强完成，train 集图像总数: 10038


## 步骤8：阶段1训练（公开数据集预训练，COCO权重起步）

In [ ]:
subprocess.run(
    ["python", "training/train.py", "--stage", "1", "--config", "training/configs/stage1_public_pretrain.yaml"],
    check=True,
)

## 步骤9：评估

In [ ]:
subprocess.run(
    [
        "python",
        "training/eval.py",
        "--weights",
        "runs/detect/training/runs/stage1/weights/best.pt",
        "--data",
        "data/dataset/data.yaml",
    ],
    check=True,
)

import pandas as pd
pd.read_csv("training/eval_report.csv")


## 步骤10（可选）：启动推理服务本地测试

In [ ]:
import subprocess
import time

weights = Path("runs/detect/training/runs/stage1/weights/best.pt")
target = Path("system/backend/models/best.pt")
target.parent.mkdir(parents=True, exist_ok=True)
target.write_bytes(weights.read_bytes())

proc = subprocess.Popen(
    ["uvicorn", "main:app", "--reload", "--port", "8000"],
    cwd="system/backend",
)
time.sleep(3)
print("推理服务: http://127.0.0.1:8000/detect")


## 可选：打包训练结果

In [ ]:
import shutil

output = Path("/kaggle/working/training_runs.zip")
shutil.make_archive(
    output.with_suffix("").as_posix(),
    "zip",
    "runs/detect/training/runs",
)
print(f"已生成: {output}")